# Pipecat Cascade + LangSmith

Build the voice agent in the notebook. The only hidden pieces are local mic/speaker transport, audio recording, and the runner.

In [ ]:
import os
import uuid

from dotenv import load_dotenv

load_dotenv()

PROJECT = "voice-workshop-cascade"
STT_MODEL = os.getenv("PIPECAT_STT_MODEL", "gpt-4o-mini-transcribe")
LLM_MODEL = os.getenv("PIPECAT_LLM_MODEL", "gpt-4o-mini")
TTS_VOICE = os.getenv("PIPECAT_TTS_VOICE", "alloy")

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

In [ ]:
from langsmith.integrations.pipecat import configure_pipecat, set_thread_id

conversation_id = str(uuid.uuid4())
set_thread_id(conversation_id)

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_TRACING_MODE", "otel")

span_processor = configure_pipecat(
    project=PROJECT,
    service_name="workshop-cascade",
    llm_span_kind="chain",
)
assert span_processor is not None

In [ ]:
from pipecat.audio.vad.silero import SileroVADAnalyzer
from pipecat.frames.frames import TTSSpeakFrame
from pipecat.pipeline.pipeline import Pipeline
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import (
    LLMContextAggregatorPair,
    LLMUserAggregatorParams,
)
from pipecat.services.openai.stt import OpenAISTTService
from pipecat.services.openai.tts import OpenAITTSService

from voice_demo.pipecat_with_langgraph.graph import GREETING, SYSTEM_PROMPT, build_graph
from voice_demo.pipecat_with_langgraph.langgraph_llm_service import LangGraphLLMService
from voice_demo.workshop import local_transport, recorder, run_task

transport = local_transport()
stt = OpenAISTTService(settings=OpenAISTTService.Settings(model=STT_MODEL))
llm = LangGraphLLMService(
    graph=build_graph(SYSTEM_PROMPT),
    settings=LangGraphLLMService.Settings(
        model=LLM_MODEL,
        system_instruction=SYSTEM_PROMPT,
    ),
)
tts = OpenAITTSService(settings=OpenAITTSService.Settings(voice=TTS_VOICE))

context = LLMContext()
context_aggregator = LLMContextAggregatorPair(
    context,
    user_params=LLMUserAggregatorParams(vad_analyzer=SileroVADAnalyzer()),
)
audiobuffer = recorder(span_processor, conversation_id)

pipeline = Pipeline(
    [
        transport.input(),
        stt,
        context_aggregator.user(),
        llm,
        tts,
        transport.output(),
        audiobuffer,
        context_aggregator.assistant(),
    ]
)

In [ ]:
# Uses your local mic and speaker. Stop the cell to end the session.
await audiobuffer.start_recording()
await run_task(
    pipeline,
    conversation_id,
    before_run=[TTSSpeakFrame(text=GREETING, append_to_context=True)],
)